# Export a saved full QLoRA run to GGUF — fresh Kaggle session

This is a standalone **conversion-only** notebook. Run its cells from top to bottom (or **Run All** here); it initializes every required variable, validates your attached full-run output, reloads its saved adapter/tokenizer, then exports Q4_K_M, Q5_K_M, and Q8_0. It contains no training, dataset preparation, judge calls, or generation tests. Do not use Run All on the training notebook just to export.

1. Import this file as a new Kaggle notebook and keep inputs/output private.
2. Under **Add Input**, attach the output of the completed **full training run**. It must include the run reports and the complete `adapter/` folder with weights and tokenizer, not only JSON reports, a ZIP that has not been extracted, a search result, or a GGUF file. The training CSV is not needed.
3. Select one NVIDIA GPU and enable Internet for installing dependencies/downloading the base model and conversion tooling. Optional access credentials go in the Kaggle secret `HF_TOKEN`; do not put them in the notebook.
4. In configuration, leave `SOURCE_RUN_DIR=""` if exactly one complete full run is attached, or set it to the **run folder** containing `run_config.json`, `run_summary.json`, and `adapter/`. Select the desired formats.
5. Run this export notebook once. Use a saved background execution if desired and download its completed Output files. A fresh Save & Run All execution does not inherit files from an earlier interactive session; it can repeat conversion. Rerunning export cells in the same session reuses valid existing formats.

The selected full-run adapter already represents the saved model selection. This notebook does not choose a different checkpoint or resume training. If only a screening/search run finished, first produce a full run or attach your existing full Colab run. Exporting does not fix grounding/escalation problems.


## 1. Install dependencies

Use a fresh GPU session. If installation requests a kernel restart, restart and continue from configuration (the cell below installation). These package pins match the training notebook; consult the source run's environment report if a saved adapter cannot load. Conversion can need substantial working disk for merged intermediates and three GGUF files. If it fails, preserve successful outputs and the error log rather than retraining.


In [ ]:
%pip install unsloth "transformers==4.57.6" "trl==0.22.2" "datasets==4.3.0"


## 2. Select the saved full-run output and requested formats


In [ ]:
import os
import json
import hashlib
from pathlib import Path

# This notebook exports on Kaggle's GPU; it does not use your local Ollama GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# The RUN FOLDER containing run_config.json, run_summary.json, and adapter/.
# Leave blank to find exactly one completed full run in the attached inputs.
SOURCE_RUN_DIR = ""
# Example: "/kaggle/input/<attached-output>/Capstone_Models/llama32_3b_best_full_01"
GGUF_QUANTIZATIONS = ["q4_k_m", "q5_k_m", "q8_0"]
OUTPUT_ROOT = Path("/kaggle/working/GGUF_Exports")


## 3. Initialize validation, provenance, and export helpers

Run this cell before source selection/export. These functions use local files only; model loading happens in the final cell.


In [ ]:
def validate_gguf_quantizations(quantizations):
    if not isinstance(quantizations, (list, tuple)) or not quantizations:
        raise ValueError("GGUF_QUANTIZATIONS must be a nonempty list of q4_k_m, q5_k_m, or q8_0.")
    methods = [str(value).lower() for value in quantizations]
    if len(set(methods)) != len(methods) or any(method not in {"q4_k_m", "q5_k_m", "q8_0"} for method in methods):
        raise ValueError("Choose unique GGUF quantizations from q4_k_m, q5_k_m, q8_0.")
    return methods

def collect_gguf_export(run_dir, quantization="q4_k_m"):
    """Recover only the requested quantization, validating all moves first.

    Other quantizations and intermediate F16 files stay untouched. A GGUF
    header/file-size check is not an inference or model-quality test.
    """
    import re
    import shutil

    method = validate_gguf_quantizations([quantization])[0]
    run_dir = Path(run_dir)
    destination = run_dir / "gguf"
    roots = (destination, run_dir / "gguf_gguf")
    candidates = []
    for root in roots:
        if not root.is_dir():
            continue
        for path in sorted(root.rglob("*.gguf")):
            if not re.search(r"(?:^|[._-])" + re.escape(method) + r"\.gguf$", path.name.lower()) or not path.is_file():
                continue
            if path.stat().st_size < 24:
                raise RuntimeError(f"GGUF export is too small: {path}")
            with path.open("rb") as handle:
                if handle.read(4) != b"GGUF":
                    raise RuntimeError(f"Invalid GGUF header: {path}")
            candidates.append(path)
    targets = {}
    for source in candidates:
        target = destination / source.name
        if target in targets or (target.exists() and source != target):
            raise FileExistsError(f"Conflicting GGUF exports for {target}; review both files before retrying.")
        targets[target] = source
    if candidates:
        destination.mkdir(parents=True, exist_ok=True)
    for target, source in targets.items():
        if source != target:
            shutil.move(str(source), str(target))
    return sorted(targets)

def export_gguf_variants(run_dir, quantizations, exporter, save_report):
    """Export/recover selected variants sequentially and persist partial progress.

    exporter is called only for missing formats; it must use the same selected
    full-run model and tokenizer for each format. Existing exports are reused.
    """
    methods = validate_gguf_quantizations(quantizations)
    run_dir = Path(run_dir)
    exports = {}
    for method in methods:
        files = collect_gguf_export(run_dir, method)
        if files:
            print(f"Reusing existing {method.upper()} export; conversion was not repeated.")
        else:
            exporter(method)
            files = collect_gguf_export(run_dir, method)
        if not files:
            raise RuntimeError(f"No {method.upper()} GGUF found in {run_dir / 'gguf'} or {run_dir / 'gguf_gguf'}. Inspect conversion output paths/logs.")
        exports[method] = [str(path) for path in files]
        save_report(run_dir / "gguf_export.json", {
            "schema_version": 2,
            "requested_quantizations": methods,
            "completed_quantizations": list(exports),
            "exports_complete": len(exports) == len(methods),
            "quantization": methods[0] if len(methods) == 1 else None,
            "files": [path for paths in exports.values() for path in paths],
            "exports": {key: [{"path": name, "size_bytes": Path(name).stat().st_size} for name in names] for key, names in exports.items()},
            "verification": "GGUF magic header and minimum file size; inference not tested",
        })
        for path in files:
            print(f"Saved GGUF: {path} ({path.stat().st_size / 2**30:.2f} GiB)")
    return exports


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=4, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")


def inspect_full_run(run_dir):
    run_dir = Path(run_dir).resolve()
    for name in ("run_config.json", "run_summary.json"):
        if not (run_dir / name).is_file():
            raise FileNotFoundError("SOURCE_RUN_DIR must be the run folder containing run_config.json, run_summary.json, and adapter/.")
    config = json.loads((run_dir / "run_config.json").read_text())
    summary = json.loads((run_dir / "run_summary.json").read_text())
    if config.get("run_mode") != "full" or summary.get("run_mode") != "full" or summary.get("smoke_test_only") is True:
        raise ValueError("Attach a completed FULL run. Search trials and smoke adapters are not full deployment candidates.")
    steps = summary.get("optimizer_steps")
    if type(steps) is not int or steps <= 0 or summary.get("all_recorded_losses_finite") is not True:
        raise ValueError("Full-run completion evidence is missing or losses were nonfinite; review run_summary.json.")
    adapter = run_dir / "adapter"
    if not (adapter / "adapter_config.json").is_file() or not any(
        (adapter / name).is_file() and (adapter / name).stat().st_size > 0
        for name in ("adapter_model.safetensors", "adapter_model.bin")
    ):
        raise FileNotFoundError("Saved adapter config/weights are missing. Attach the full run OUTPUT, not just reports or a GGUF file.")
    if not (adapter / "tokenizer_config.json").is_file() or not any(
        (adapter / name).is_file() for name in ("tokenizer.json", "tokenizer.model")
    ):
        raise FileNotFoundError("Saved tokenizer files are missing from adapter/. Attach the complete saved adapter folder.")
    tokenizer_config = json.loads((adapter / "tokenizer_config.json").read_text())
    template_file = adapter / "chat_template.jinja"
    saved_template = template_file.read_text() if template_file.is_file() else tokenizer_config.get("chat_template")
    if not isinstance(saved_template, str) or not saved_template.strip():
        raise ValueError("A single saved Llama chat template is required; restore the original tokenizer/template before exporting.")
    run_name = config.get("run_name") or run_dir.name
    if not isinstance(run_name, str) or not run_name or Path(run_name).name != run_name or run_name in {".", ".."}:
        raise ValueError("Invalid source run name.")
    max_seq_length = config.get("max_seq_length")
    if type(max_seq_length) is not int or max_seq_length <= 0:
        raise ValueError("run_config.json must record a positive max_seq_length.")
    return {"source_run_dir": str(run_dir), "run_name": run_name,
            "adapter_dir": str(adapter), "max_seq_length": max_seq_length, "saved_chat_template": saved_template,
            "source_report_sha256": {name: sha256_file(run_dir / name) for name in ("run_config.json", "run_summary.json")},
            "source_adapter_sha256": {str(path.relative_to(adapter)): sha256_file(path)
                                       for path in sorted(adapter.rglob("*")) if path.is_file()}}


def find_full_run(source_run_dir, input_root="/kaggle/input"):
    if source_run_dir:
        return inspect_full_run(source_run_dir)
    found = []
    for report in sorted(Path(input_root).rglob("run_summary.json")):
        try:
            found.append(inspect_full_run(report.parent))
        except (ValueError, FileNotFoundError):
            continue
    if len(found) != 1:
        names = [item["source_run_dir"] for item in found]
        raise ValueError(f"Expected one completed full run with saved adapter/tokenizer; found {len(found)}: {names}. Attach its output or set SOURCE_RUN_DIR explicitly.")
    return found[0]


def prepare_export(source_run_dir, output_root, input_root="/kaggle/input"):
    provenance = find_full_run(source_run_dir, input_root)
    source = Path(provenance["source_run_dir"])
    output = (Path(output_root) / provenance["run_name"]).resolve()
    if output.is_relative_to(source) or source.is_relative_to(output):
        raise ValueError("Export output must be separate from the read-only source run.")
    manifest = output / "source_provenance.json"
    if manifest.exists():
        previous = json.loads(manifest.read_text())
        if any(previous.get(key) != provenance[key] for key in ("source_report_sha256", "source_adapter_sha256")):
            raise ValueError("This output directory belongs to different source weights/reports. Choose a new OUTPUT_ROOT.")
    elif output.exists() and any(output.iterdir()):
        raise FileExistsError("Output is not empty and has no source manifest. Choose a new OUTPUT_ROOT to avoid mixing models.")
    output.mkdir(parents=True, exist_ok=True)
    save_json(manifest, provenance)
    return {**provenance, "output_dir": str(output)}


def export_saved_adapter(context, quantizations, model_loader, hf_token=None):
    methods = validate_gguf_quantizations(quantizations)
    output = Path(context["output_dir"])
    missing = [method for method in methods if not collect_gguf_export(output, method)]
    if missing:
        # Loading saved PEFT weights also loads the base model. Do not create a
        # new adapter or run a trainer; preserve the saved tokenizer/template.
        model, tokenizer = model_loader(
            model_name=context["adapter_dir"], max_seq_length=context["max_seq_length"],
            dtype=None, load_in_4bit=True, token=hf_token,
        )
        # Restore the authoritative saved template if the loader supplied defaults.
        tokenizer.chat_template = context["saved_chat_template"]
        def exporter(method):
            model.save_pretrained_gguf(str(output / "gguf"), tokenizer, quantization_method=method)
    else:
        def exporter(method):
            raise RuntimeError("A recovered export unexpectedly became unavailable; inspect output files.")
    return export_gguf_variants(output, methods, exporter, save_json)


## 4. Validate and identify source/output folders

Inputs stay read-only. A source manifest binds recovered exports to the same adapter/tokenizer/report hashes. Existing outputs from a different model are rejected. The writable output directory is separate from the attached training run.


In [ ]:
# Validate all selections and source files before allocating model memory.
validate_gguf_quantizations(GGUF_QUANTIZATIONS)
export_context = prepare_export(SOURCE_RUN_DIR, OUTPUT_ROOT)
print("Source full run:", export_context["source_run_dir"])
print("Saved adapter:", export_context["adapter_dir"])
print("Export output:", export_context["output_dir"])
print("Requested formats:", GGUF_QUANTIZATIONS)


## 5. Reload the saved adapter and export

This cell allocates GPU memory on Kaggle. It reloads the adapter only if a requested format is missing, then exports missing formats sequentially from that same model. The original saved chat template is retained. Header/size checks are basic artifact validation, not proof of model quality. No base-model-only fallback is used if adapter loading fails.


In [ ]:
# Import Unsloth before torch/transformers. These are the only model operations.
from unsloth import FastLanguageModel

hf_token = os.getenv("HF_TOKEN") or None
if hf_token is None:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass  # Public models may work without a token; never print a secret.

exports = export_saved_adapter(
    export_context, GGUF_QUANTIZATIONS,
    FastLanguageModel.from_pretrained, hf_token=hf_token,
)
print("Exported/recovered formats:", list(exports))
print("Download gguf/ plus gguf_export.json and source_provenance.json from:", export_context["output_dir"])
print("Conversion finished. Local inference, quality, latency, and memory comparisons are separate checks.")


## 6. Download the output

Files are under `/kaggle/working/GGUF_Exports/<source-run-name>/`:

- `gguf/*.Q4_K_M.gguf`, `gguf/*.Q5_K_M.gguf`, `gguf/*.Q8_0.gguf` for selected completed formats (exact filename prefixes depend on the exporter).
- `gguf_export.json`: requested/completed formats, partial/completed status, files, and sizes.
- `source_provenance.json`: source full-run reports and adapter/tokenizer hashes.

Keep the original training output as well. Local Ollama imports/inference and matched quality/latency/memory comparisons follow separately after the active local judge finishes. A completed conversion is not another training cycle and does not complete those comparisons. Actual GPU export execution remains to be verified in Kaggle.
